1. Import des librairies

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

2. Charger le dataset

In [ ]:
df = pd.read_csv("final_air_quality_dataset_6000.csv")

X = df[["Temperature_C", "Humidity_%", "MQ2_ADC"]]
y = df["Air_Quality"]

In [10]:
# Encoder les labels
encoder = LabelEncoder()
y = encoder.fit_transform(y)

3. Split Train / Validation / Test

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

4. Normalisation

In [12]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

5. Définition des modèles

In [17]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=9,
        weights="distance"
    ),

    "SVM (RBF)": SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42
    ),

    "Neural Network (MLP)": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        alpha=0.001,
        batch_size=64,
        max_iter=600,
        early_stopping=True,
        random_state=42
    )
}

6. Entraînement & Validation

In [18]:
results = {}

print("----- Résultats Validation -----\n")

for name, model in models.items():
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_val_pred)
    results[name] = acc
    print(f"{name} : Accuracy = {acc:.4f}")

----- Résultats Validation -----

Logistic Regression : Accuracy = 0.6078
KNN : Accuracy = 0.9622
SVM (RBF) : Accuracy = 0.9489
Random Forest : Accuracy = 0.9967
Neural Network (MLP) : Accuracy = 0.9633


In [ ]:
print("\n===== Détection Overfitting / Underfitting =====\n")

for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred   = model.predict(X_val)

    train_acc = accuracy_score(y_train, train_pred)
    val_acc   = accuracy_score(y_val, val_pred)

    print(f"{name}")
    print(f"  Train Accuracy : {train_acc:.4f}")
    print(f"  Val Accuracy   : {val_acc:.4f}")

    if train_acc - val_acc > 0.10:
        print(" Overfitting détecté\n")
    elif train_acc < 0.70 and val_acc < 0.70:
        print(" Underfitting détecté\n")
    else:
        print(" Bon compromis\n")



===== Détection Overfitting / Underfitting =====

Logistic Regression
  Train Accuracy : 0.5905
  Val Accuracy   : 0.6078
  ⚠️ Underfitting détecté

KNN
  Train Accuracy : 1.0000
  Val Accuracy   : 0.9622
  ✅ Bon compromis

SVM (RBF)
  Train Accuracy : 0.9405
  Val Accuracy   : 0.9489
  ✅ Bon compromis

Random Forest
  Train Accuracy : 0.9998
  Val Accuracy   : 0.9967
  ✅ Bon compromis

Neural Network (MLP)
  Train Accuracy : 0.9476
  Val Accuracy   : 0.9633
  ✅ Bon compromis



7. Choisir le meilleur modèle

In [20]:
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("\n Meilleur modèle :", best_model_name)


 Meilleur modèle : Random Forest


8. Test final

In [21]:
y_test_pred = best_model.predict(X_test)

print("\n===== Évaluation finale (Test) =====")
print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("\nClassification Report :\n",
      classification_report(y_test, y_test_pred,
                            target_names=encoder.classes_))

print("\nConfusion Matrix :\n",
      confusion_matrix(y_test, y_test_pred))



===== Évaluation finale (Test) =====
Accuracy : 0.9955555555555555

Classification Report :
               precision    recall  f1-score   support

   Dangerous       0.99      1.00      0.99       222
        Good       1.00      1.00      1.00        18
    Moderate       0.99      0.98      0.99       160
        Poor       1.00      1.00      1.00       500

    accuracy                           1.00       900
   macro avg       1.00      0.99      0.99       900
weighted avg       1.00      1.00      1.00       900


Confusion Matrix :
 [[221   0   1   0]
 [  0  18   0   0]
 [  3   0 157   0]
 [  0   0   0 500]]


9. SAUVEGARDER LE MODÈLE

In [ ]:
import joblib

# Sauvegarde du modèle
joblib.dump(best_model, "random_forest_air_quality_model.joblib")

print("Modèle sauvegardé avec succès")

joblib.dump(scaler, "scaler.joblib")
print(" Scaler sauvegardé")



Modèle sauvegardé avec succès
✅ Scaler sauvegardé
